# Day 1: I Own the Training Loop

**Goal**: Train an NLP model from scratch and debug it confidently.

**Time**: 4 hours
- 20 min: Planning
- 2.5 hrs: Coding
- 45 min: Debugging & experiments
- 25 min: Reflection

---

## Confidence Checks
- [ ] I can explain why incorrect padding or masking breaks training
- [ ] I can change embedding size or max sequence length without panic
- [ ] I know exactly where NaNs come from

## Task 1: Text Pipeline (45 min)

**Dataset**: IMDb (5k samples)

Build:
- Whitespace tokenizer
- Vocabulary (PAD=0, UNK=1)
- Padding and attention masks
- Custom Dataset and DataLoader

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import from nlp_utils package
from nlp_utils import (
    tokenize,
    Vocabulary,
    ReviewDataSet,
    collate_fn,
    extract_imdb_sample_as_dict
)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from datasets import load_dataset

In [5]:
ds = load_dataset("stanfordnlp/imdb")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 260220.97 examples/s]


In [19]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
# Extract sampled splits as dictionaries
train, test = extract_imdb_sample_as_dict(ds, train_size=5000, test_size=1000)

In [175]:
type(train)

dict

In [176]:
sample_text = "Hello! This is a simple example. Let's tokenize this text."
    
tokens = tokenize(sample_text)

In [ ]:
# Create and build vocabulary
vocab = Vocabulary(tokenizer=tokenize)
vocab.build_from_texts(train['text'])

In [ ]:
# Create datasets
train_dataset = ReviewDataSet(target=train, vocab=vocab)
test_dataset = ReviewDataSet(target=test, vocab=vocab)

In [200]:
# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training
    collate_fn=collate_fn,
    num_workers=0          # Set to 0 for debugging, increase for speed
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # Don't shuffle for testing
    collate_fn=collate_fn,
    num_workers=0
)




In [202]:
batch=next(iter(train_loader))